<a href="https://colab.research.google.com/github/salavii/SOP-Generator-Fine-tuning/blob/main/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Llama 3.1 8B for SOP Generation

This notebook fine-tunes Llama 3.1 8B using QLoRA (4-bit quantization) for efficient training on limited GPU resources.

**Training approach:**
- Base model: Llama 3.1 8B Instruct
- Method: QLoRA (Low-Rank Adaptation with 4-bit quantization)
- Dataset: 400 training + 100 validation SOPs
- Hardware: Google Colab T4 GPU

In [1]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✓ GPU detected: {gpu_name}")
    print(f"✓ GPU memory: {gpu_memory:.2f} GB")
else:
    print("✗ No GPU detected!")
    print("Go to Runtime → Change runtime type → Select GPU")

✓ GPU detected: Tesla T4
✓ GPU memory: 14.74 GB


## Install Required Libraries

Install HuggingFace libraries for fine-tuning:
- `transformers`: Model loading and training
- `peft`: Parameter-Efficient Fine-Tuning (QLoRA)
- `bitsandbytes`: 4-bit quantization
- `accelerate`: Distributed training utilities
- `trl`: Transformer Reinforcement Learning (for SFTTrainer)

In [2]:
# Install dependencies (minimal, stable)
!pip install -q transformers
!pip install -q peft
!pip install -q accelerate
!pip install -q datasets
!pip install -q trl

print("✓ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 14.3 MB/s eta 0:00:00
✓ Dependencies installed!


## Upload Training Data

Upload the JSONL files created in the previous notebook.

In [3]:
# Upload train and validation data
from google.colab import files

print("Upload train.jsonl and val.jsonl:")
uploaded = files.upload()

# Verify files
import os
if 'train.jsonl' in uploaded and 'val.jsonl' in uploaded:
    train_size = os.path.getsize('train.jsonl') / (1024**2)
    val_size = os.path.getsize('val.jsonl') / (1024**2)
    print(f"\n✓ train.jsonl uploaded ({train_size:.2f} MB)")
    print(f"✓ val.jsonl uploaded ({val_size:.2f} MB)")
else:
    print("\n✗ Missing files! Please upload both train.jsonl and val.jsonl")

Upload train.jsonl and val.jsonl:


Saving train.jsonl to train.jsonl
Saving val.jsonl to val.jsonl

✓ train.jsonl uploaded (1.64 MB)
✓ val.jsonl uploaded (0.41 MB)


## Load and Prepare Dataset

Load JSONL files into HuggingFace Dataset format for training.

In [9]:
# Load datasets
from datasets import load_dataset

# Load JSONL files
train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
val_dataset = load_dataset('json', data_files='val.jsonl', split='train')

print(f"✓ Training examples: {len(train_dataset)}")
print(f"✓ Validation examples: {len(val_dataset)}")


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

✓ Training examples: 400
✓ Validation examples: 100


In [4]:

# Load Mistral 7B (FP16 - no quantization)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading {MODEL_NAME}...")
print("This may take 5-10 minutes...\n")

# Load model in FP16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✓ Model loaded successfully!")
print(f"✓ Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading mistralai/Mistral-7B-Instruct-v0.3...
This may take 5-10 minutes...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✓ Model loaded successfully!
✓ Model size: 14.50 GB


## Configure LoRA Adapters

Setup LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning:
- Only trains small adapter layers (~1% of parameters)
- Much faster and memory-efficient
- Achieves similar quality to full fine-tuning

In [13]:
# Configure LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                      # Rank of adaptation matrices
    lora_alpha=32,             # Scaling factor
    target_modules=[           # Layers to apply LoRA
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 13,631,488 || all params: 7,261,655,040 || trainable%: 0.1877


## Training Configuration

Define training hyperparameters:
- Batch size, learning rate, epochs
- Gradient accumulation for effective larger batches
- Evaluation and saving strategy

In [14]:
# Cell 8: Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sop-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

print("✓ Training configuration ready!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

✓ Training configuration ready!
  Epochs: 3
  Batch size: 1
  Learning rate: 0.0002


## Initialize Trainer

Setup the SFTTrainer (Supervised Fine-Tuning Trainer) which handles:
- Training loop
- Evaluation
- Checkpointing
- Gradient updates

In [16]:
# Cell 9: Initialize trainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    dataset_text_field="messages",
    max_seq_length=1024,
)

print("✓ Trainer initialized!")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'